In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
import sys
from pyspark.sql import SparkSession

# 1. Environment Overrides
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/openjdk-17.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable

# 2. Hardened Spark 4.1.1 Session
spark = SparkSession.builder \
    .appName("SilverTransformations") \
    .config("spark.jars.packages", (
        "org.apache.hadoop:hadoop-azure:3.4.1,"
        "com.microsoft.azure:azure-storage:8.6.6,"
        "io.delta:delta-spark_2.13:4.1.0"
    )) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.azure.account.auth.type", "SharedKey") \
    .config("spark.hadoop.fs.abfss.impl", "org.apache.hadoop.fs.azurebfs.SecureAzureBlobFileSystem") \
    .config("spark.hadoop.fs.azure.always.use.https", "true") \
    .getOrCreate()

# 3. Azure Key (Ensure your .env is loaded or export AZURE_STORAGE_KEY first)
storage_account_name = "coinmarketcapapi"
container_name = "coin-market-cap-api-data"
storage_account_key = os.getenv("AZURE_STORAGE_KEY")

# Set the key on the DFS endpoint
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

print(f"🚀 Spark {spark.version} Ready for ADLS Gen2!")

:: loading settings :: url = jar:file:/Users/as-mac-1250/crypto-market-intelligence/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/as-mac-1250/.ivy2.5.2/cache
The jars for the packages stored in: /Users/as-mac-1250/.ivy2.5.2/jars
org.apache.hadoop#hadoop-azure added as a dependency
com.microsoft.azure#azure-storage added as a dependency
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3a9bc0b4-86e4-49df-8125-a16c3b0454d1;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.4.1 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.2 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.3.0 in central
	found org.eclipse.jetty#jetty-util-aja

🚀 Spark 4.1.1 Ready for ADLS Gen2!


In [3]:
# --- 2. PATH DEFINITIONS ---
# Updated to match your folder structure in Screenshot 3:23:46 PM
bronze_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"

silver_quotes_path = f"{silver_base_path}/crypto_quotes"
silver_global_path = f"{silver_base_path}/global_metrics"
silver_historical_path = f"{silver_base_path}/historical_prices"

# --- 3. LOAD BRONZE DATA ---
# We read as JSON and use recursive lookup to grab files from both cmc and coingecko subfolders
bronze_df = spark.read.option("recursiveFileLookup", "true") \
    .json(bronze_base_path)

# Creating temporary aliases to match your transformation logic names
# Your ingestion script wraps everything in 'payload', 'ingested_at', 'source', etc.
bronze_df = bronze_df.withColumnRenamed("ingested_at", "event_timestamp")

print(f"✅ Bronze JSON data loaded. Total rows: {bronze_df.count()}")

✅ Bronze JSON data loaded. Total rows: 5


In [4]:
from pyspark.sql.functions import col, from_json, explode, get_json_object, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

# --- A. PROCESS REAL-TIME QUOTES (From Listings Endpoint) ---
# Note: payload is an array of coins
cmc_listings_schema = ArrayType(StructType([
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("symbol", StringType()),
    StructField("quote", StructType([
        StructField("USD", StructType([
            StructField("price", DoubleType()),
            StructField("volume_24h", DoubleType()),
            StructField("market_cap", DoubleType())
        ]))
    ]))
]))

silver_quotes = bronze_df.filter(col("endpoint_type") == "listings") \
    .withColumn("parsed_payload", from_json(col("payload"), cmc_listings_schema)) \
    .withColumn("coin", explode(col("parsed_payload"))) \
    .select(
        col("coin.id").alias("coin_id"),
        col("coin.symbol"),
        col("coin.name"),
        col("coin.quote.USD.price").alias("price_usd"),
        col("coin.quote.USD.volume_24h").alias("volume_24h"),
        col("coin.quote.USD.market_cap").alias("market_cap"),
        col("event_timestamp"),
        current_timestamp().alias("silver_processing_timestamp")
    )

silver_global = bronze_df.filter(col("endpoint_type") == "global") \
    .select(
        get_json_object(col("payload"), "$.quote.USD.total_market_cap").cast("double").alias("total_market_cap"),
        get_json_object(col("payload"), "$.quote.USD.total_volume_24h").cast("double").alias("total_volume_24h"),
        get_json_object(col("payload"), "$.btc_dominance").cast("double").alias("btc_dominance"),
        col("event_timestamp")
    )

In [5]:
# --- C. PROCESS HISTORICAL BACKFILL ---
# Schema for CoinGecko's nested arrays: [[timestamp, value], ...]
coingecko_schema = StructType([
    StructField("prices", ArrayType(ArrayType(DoubleType()))),
    StructField("market_caps", ArrayType(ArrayType(DoubleType()))),
    StructField("total_volumes", ArrayType(ArrayType(DoubleType())))
])

# 1. Select the raw rows for historical data
historical_df = bronze_df.filter(col("endpoint_type") == "historical_backfill")

# 2. Parse payload and explode prices
# Note: Using col("payload") directly instead of raw_content
silver_historical = historical_df.withColumn("parsed", from_json(col("payload"), coingecko_schema)) \
    .withColumn("exploded_prices", explode(col("parsed.prices"))) \
    .select(
        col("coin_id"), # Injected by your ingestion script
        (col("exploded_prices")[0] / 1000).cast("timestamp").alias("price_timestamp"),
        col("exploded_prices")[1].alias("price_usd"),
        col("source"),
        current_timestamp().alias("silver_processing_timestamp")
    )

In [6]:
# --- D. SAVE AS DELTA TABLES ---
print("Writing Silver tables to Azure...")

# Define write function to keep it clean
def save_to_silver(df, path):
    df.write.format("delta") \
      .mode("overwrite") \
      .option("mergeSchema", "true") \
      .save(path)

save_to_silver(silver_quotes, silver_quotes_path)
save_to_silver(silver_global, silver_global_path)
save_to_silver(silver_historical, silver_historical_path)

print("✅ Silver Layer Processing Complete!")

Writing Silver tables to Azure...


26/04/16 17:40:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✅ Silver Layer Processing Complete!


In [7]:
# --- E. VERIFY SAMPLES ---
print("Silver Quotes Sample:")
spark.read.format("delta").load(silver_quotes_path).show(5)

print("Silver Historical Sample:")
spark.read.format("delta").load(silver_historical_path).show(5)

print("Silver Global Metrics Sample:")
spark.read.format("delta").load(silver_global_path).show(5)

Silver Quotes Sample:
+-------+------+-----------+------------------+--------------------+--------------------+--------------------+---------------------------+
|coin_id|symbol|       name|         price_usd|          volume_24h|          market_cap|     event_timestamp|silver_processing_timestamp|
+-------+------+-----------+------------------+--------------------+--------------------+--------------------+---------------------------+
|      1|   BTC|    Bitcoin| 74650.23175439381|3.824245741420280...|1.494242858481986...|2026-04-16T12:09:...|       2026-04-16 17:40:...|
|   1027|   ETH|   Ethereum| 2336.739913382459|1.731557177085352...|2.820233172855268E11|2026-04-16T12:09:...|       2026-04-16 17:40:...|
|    825|  USDT|Tether USDt|0.9999969223111252|1.189821368171384E11|1.857417203827361E11|2026-04-16T12:09:...|       2026-04-16 17:40:...|
|     52|   XRP|        XRP|1.4160440465791133| 3.374480767302941E9|8.718537919186487E10|2026-04-16T12:09:...|       2026-04-16 17:40:...|
|   1

In [8]:
spark.stop()